# Milestone 6 — Rule Proposer Demo

Pipeline:
1. Load stored InvestigationRecords via MemoryStore
2. Run pattern detection (48h window, min_count=2)
3. For each PatternFinding, call `propose_rule()`
4. Save each proposal to ProposalStore
5. Print all proposals in readable format

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from src.sentinel.pattern_detection import run_pattern_detection
from src.sentinel.rule_proposer import propose_rule, ProposalStore
from src.sentinel.memory_store import MemoryStore

store = MemoryStore()
proposal_store = ProposalStore()

all_records = store._load_all()
print(f"Total investigation records in store: {len(all_records)}")

Total investigation records in store: 5


In [3]:
# Run pattern detection across all stored records (wide window to catch seeded data)
findings = run_pattern_detection(time_window_hours=8760, min_count=2, store=store)
print(f"PatternFindings detected: {len(findings)}")
for f in findings:
    print(f"  {f.pattern_tags}  count={f.count}  type={f.finding_type}")

PatternFindings detected: 22
  ['new_account']  count=4  type=novel_pattern
  ['ring_fraud']  count=3  type=novel_pattern
  ['identity_fraud']  count=3  type=novel_pattern
  ['multi_domain_convergence']  count=3  type=novel_pattern
  ['identity_fraud', 'ring_fraud']  count=3  type=novel_pattern
  ['new_account', 'ring_fraud']  count=3  type=novel_pattern
  ['multi_domain_convergence', 'ring_fraud']  count=3  type=novel_pattern
  ['identity_fraud', 'new_account']  count=3  type=novel_pattern
  ['identity_fraud', 'multi_domain_convergence']  count=3  type=novel_pattern
  ['multi_domain_convergence', 'new_account']  count=3  type=novel_pattern
  ['account_factory']  count=2  type=novel_pattern
  ['high_confidence_block']  count=2  type=novel_pattern
  ['account_factory', 'ring_fraud']  count=2  type=novel_pattern
  ['high_confidence_block', 'ring_fraud']  count=2  type=novel_pattern
  ['account_factory', 'identity_fraud']  count=2  type=novel_pattern
  ['high_confidence_block', 'identity_

In [4]:
# Generate a rule proposal for the top 3 findings by count
top_findings = sorted(findings, key=lambda f: f.count, reverse=True)[:3]

proposals = []
for finding in top_findings:
    proposal = propose_rule(finding)
    proposal_store.save(proposal)
    proposals.append(proposal)
    print(f"Proposed: {proposal.rule_name} ({proposal.rule_type})")

print(f"\nTotal proposals generated: {len(proposals)}")

Proposed: new_account_verification_enhancement (hard_rule)


Proposed: monitor_ring_fraud_activity (sentinel_filter)


Proposed: block_identity_fraud (hard_rule)

Total proposals generated: 3


In [5]:
# Print all proposals in readable format
print("=" * 70)
for p in proposals:
    print(f"Rule name   : {p.rule_name}")
    print(f"Rule type   : {p.rule_type}")
    print(f"Pattern tags: {p.pattern_tags}")
    print(f"Count       : {p.pattern_count}  |  Finding type: {p.pattern_finding_type}")
    print(f"Description : {p.description}")
    print(f"Change      : {p.proposed_change}")
    print(f"Impact      : {p.expected_impact}")
    print(f"Risk        : {p.risk}")
    print(f"Status      : {p.status}")
    print("-" * 70)

Rule name   : new_account_verification_enhancement
Rule type   : hard_rule
Pattern tags: ['new_account']
Count       : 4  |  Finding type: novel_pattern
Description : Enhance verification processes for new account patterns to reduce potential fraud.
Change      : {'condition': 'transaction.is_new_account and transaction.amount > 1000', 'verdict': 'step_up', 'rationale': 'New accounts with high transaction amounts should undergo additional verification to prevent fraud.'}
Impact      : This change will increase scrutiny on new accounts making large transactions, reducing potential fraud.
Risk        : There is a risk of increased friction for legitimate new users making large transactions.
Status      : pending
----------------------------------------------------------------------
Rule name   : monitor_ring_fraud_activity
Rule type   : sentinel_filter
Pattern tags: ['ring_fraud']
Count       : 3  |  Finding type: novel_pattern
Description : Monitor 'ring_fraud' patterns for any increase

In [6]:
# Show pending proposals from store
pending = proposal_store.load_pending()
print(f"Pending proposals in store: {len(pending)}")

Pending proposals in store: 52
